# RetailRocket - Dummy and Baseline Experiments

Objetivo: criar uma primeira esteira de recomendacao com split temporal, dummy de popularidade e baseline colaborativo item-item, com metricas de ranking.

## Etapa 1 - Imports e configuracoes

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import mlflow

RANDOM_STATE = 42
TOP_K = 10
EVENT_WEIGHT = {"view": 1.0, "addtocart": 3.0, "transaction": 5.0}

## MLflow Setup - Rastreamento de Experimentos

Inicializamos MLflow para registrar cada variação de baseline com seus parâmetros e métricas.

In [ ]:
# Configurar backend local do MLflow
MLFLOW_TRACKING_URI = Path("../mlruns").absolute()
mlflow.set_tracking_uri(f"file:{MLFLOW_TRACKING_URI}")
mlflow.set_experiment("baseline_experiments")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('baseline_experiments').name}")

## Etapa 2 - Carga e preparo dos dados

Usamos apenas interacoes implicitas (view, addtocart, transaction) com peso por tipo de evento.

In [3]:
DATA_PATH = Path("../data/raw/retailrocket/events.csv")
assert DATA_PATH.exists(), f"Arquivo nao encontrado: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df = df[df["event"].isin(EVENT_WEIGHT.keys())].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
df["weight"] = df["event"].map(EVENT_WEIGHT).astype(float)

df = df.sort_values("timestamp").reset_index(drop=True)
df[["visitorid", "itemid"]] = df[["visitorid", "itemid"]].astype("int64")

print(df.head())
print("Interacoes:", len(df))
print("Usuarios:", df["visitorid"].nunique())
print("Itens:", df["itemid"].nunique())

                         timestamp  visitorid      event  itemid  \
0 2015-05-03 03:00:04.384000+00:00     693516  addtocart  297662   
1 2015-05-03 03:00:11.289000+00:00     829044       view   60987   
2 2015-05-03 03:00:13.048000+00:00     652699       view  252860   
3 2015-05-03 03:00:24.154000+00:00    1125936       view   33661   
4 2015-05-03 03:00:26.228000+00:00     693516       view  297662   

   transactionid  weight  
0            NaN     3.0  
1            NaN     1.0  
2            NaN     1.0  
3            NaN     1.0  
4            NaN     1.0  
Interacoes: 2756101
Usuarios: 1407580
Itens: 235061


## Etapa 3 - Split temporal (treino, validacao, teste)

Split temporal evita data leakage em recomendacao.

In [4]:
n = len(df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

print("Train:", train_df["timestamp"].min(), "->", train_df["timestamp"].max(), "|", len(train_df))
print("Valid:", valid_df["timestamp"].min(), "->", valid_df["timestamp"].max(), "|", len(valid_df))
print("Test:", test_df["timestamp"].min(), "->", test_df["timestamp"].max(), "|", len(test_df))

Train: 2015-05-03 03:00:04.384000+00:00 -> 2015-08-02 23:30:53.300000+00:00 | 1929270
Valid: 2015-08-02 23:30:55.147000+00:00 -> 2015-08-25 22:30:59.619000+00:00 | 413415
Test: 2015-08-25 22:31:01.629000+00:00 -> 2015-09-18 02:59:47.788000+00:00 | 413416


## Etapa 4 - Funcoes de avaliacao

Metricas: Recall@K, HitRate@K, NDCG@K e MAP@K.

In [5]:
def build_user_items(frame: pd.DataFrame) -> dict:
    grouped = frame.groupby("visitorid")["itemid"].apply(set)
    return grouped.to_dict()


def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    hits = len(set(rec_k) & relevant)
    return hits / len(relevant)


def hitrate_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    rec_k = set(recommended[:k])
    return 1.0 if len(rec_k & relevant) > 0 else 0.0


def ndcg_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / math.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg


def map_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            hits += 1
            precision_sum += hits / i
    return precision_sum / min(len(relevant), k)


def evaluate_model(user_recs: dict[int, list[int]], truth: dict[int, set[int]], k: int = 10) -> dict[str, float]:
    users = [u for u in truth.keys() if u in user_recs]
    if not users:
        return {"recall@k": 0.0, "hitrate@k": 0.0, "ndcg@k": 0.0, "map@k": 0.0}

    recalls, hits, ndcgs, maps = [], [], [], []
    for user in users:
        recs = user_recs[user]
        rel = truth[user]
        recalls.append(recall_at_k(recs, rel, k))
        hits.append(hitrate_at_k(recs, rel, k))
        ndcgs.append(ndcg_at_k(recs, rel, k))
        maps.append(map_at_k(recs, rel, k))

    return {
        "recall@k": float(np.mean(recalls)),
        "hitrate@k": float(np.mean(hits)),
        "ndcg@k": float(np.mean(ndcgs)),
        "map@k": float(np.mean(maps)),
    }

## Etapa 5 - Modelo dummy (Most Popular)

In [ ]:
train_user_items = build_user_items(train_df)
valid_user_items = build_user_items(valid_df)

popular_items = (
    train_df.groupby("itemid")["weight"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

# Iniciar run do MLflow para Dummy Baseline
with mlflow.start_run(run_name="dummy_most_popular"):
    # Registrar parâmetros
    mlflow.log_param("model_type", "dummy_most_popular")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("train_size", len(train_df))
    mlflow.log_param("valid_size", len(valid_df))
    mlflow.log_param("event_weights", str(EVENT_WEIGHT))
    
    # Executar modelo
    dummy_recs = {}
    for user, seen_items in train_user_items.items():
        recs = [item for item in popular_items if item not in seen_items][:TOP_K]
        dummy_recs[user] = recs
    
    # Avaliar e registrar métricas
    dummy_metrics = evaluate_model(dummy_recs, valid_user_items, k=TOP_K)
    for metric_name, metric_value in dummy_metrics.items():
        mlflow.log_metric(metric_name, metric_value)
    
    print("Dummy metrics:", dummy_metrics)
    print(f"Run ID: {mlflow.active_run().info.run_id}")

KeyboardInterrupt: 

## Etapa 6 - Baseline item-item (co-ocorrencia com similaridade cosseno)

Limitacao conhecida: nao funciona para usuarios novos (cold start).

Baseline colaborativo que calcula similaridade entre itens com base no padrao de consumo dos usuarios.Ideia: se dois itens foram consumidos por usuarios parecidos, eles sao similares entre si.

In [ ]:
# Agregacao: soma dos pesos de interacao por par (usuario, item)
user_item_matrix = (
    train_df.groupby(["visitorid", "itemid"])
    .agg(score=("weight", "sum"))
    .reset_index()
)

# Mapeamento: conversao de IDs reais para indices de matriz
user_ids = user_item_matrix["visitorid"].unique().tolist()
item_ids = user_item_matrix["itemid"].unique().tolist()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: j for j, it in enumerate(item_ids)}
idx_to_item = {j: it for it, j in item_to_idx.items()}

# Construcao da matriz usuario-item: linhas sao usuarios, colunas sao itens
mat = np.zeros((len(user_ids), len(item_ids)), dtype=np.float32)
for _, row in user_item_matrix.iterrows():
    mat[user_to_idx[row["visitorid"]], item_to_idx[row["itemid"]]] = row["score"]

# Similaridade cosseno entre itens: mat.T transforma cada item em vetor de usuarios
# Resultado: matriz item-item onde cada celula eh a similaridade entre dois itens
item_sim = cosine_similarity(mat.T)
np.fill_diagonal(item_sim, 0.0)  # Evita recomendar o mesmo item (auto-similaridade)

def recommend_item_knn_for_user(user: int, top_k: int = 10) -> list[int]:
    """Recomenda itens com base em similaridade item-item.
    
    Passos:
    1. Se usuario novo (nao treino), retorna lista vazia (cold start).
    2. Calcula score para cada item como produto: vetor_usuario @ matriz_similaridade
    3. Itens similares aos ja consumidos recebem scores mais altos.
    4. Retorna Top-K excluindo ja vistos.
    """
    if user not in user_to_idx:
        return []  # Problema de cold start: usuario novo nao pode ser recomendado
    uidx = user_to_idx[user]
    user_profile = mat[uidx]  # Historico de interacoes do usuario
    seen = set(np.where(user_profile > 0)[0].tolist())  # Itens ja consumidos

    # Produto vetor_usuario @ matriz_similaridade: score para cada item
    scores = user_profile @ item_sim
    ranked_idx = np.argsort(-scores)  # Ordena do maior para menor score

    recs = []
    for idx in ranked_idx:
        if idx in seen:  # Pula itens ja interagidos
            continue
        recs.append(idx_to_item[idx])
        if len(recs) >= top_k:
            break  # Para quando atinge Top-K
    return recs

# Iniciar run do MLflow para Item-Item Baseline
with mlflow.start_run(run_name="baseline_item_item"):
    # Registrar parâmetros
    mlflow.log_param("model_type", "item_item_collaborative")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("similarity_metric", "cosine")
    mlflow.log_param("train_users", len(user_ids))
    mlflow.log_param("train_items", len(item_ids))
    mlflow.log_param("train_size", len(train_df))
    mlflow.log_param("valid_size", len(valid_df))
    mlflow.log_param("event_weights", str(EVENT_WEIGHT))
    
    # Executar modelo
    baseline_recs = {user: recommend_item_knn_for_user(user, top_k=TOP_K) for user in train_user_items.keys()}
    
    # Avaliar e registrar métricas
    baseline_metrics = evaluate_model(baseline_recs, valid_user_items, k=TOP_K)
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(metric_name, metric_value)
    
    # Armazenar recomendações como artefato (amostra)
    recs_sample = {str(u): baseline_recs[u][:5] for u in list(train_user_items.keys())[:10]}
    import json
    with open("baseline_recs_sample.json", "w") as f:
        json.dump(recs_sample, f, indent=2)
    mlflow.log_artifact("baseline_recs_sample.json")
    
    print("Item-item baseline metrics:", baseline_metrics)
    print(f"Run ID: {mlflow.active_run().info.run_id}")

## Etapa 7 - Comparacao dos resultados

In [ ]:
# Comparar resultados dos runs via MLflow
experiment = mlflow.get_experiment_by_name("baseline_experiments")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Selecionar apenas colunas relevantes e ordenar por NDCG
comparison_df = runs_df[
    ["tags.mlflow.runName", "params.model_type", "metrics.recall@k", "metrics.hitrate@k", "metrics.ndcg@k", "metrics.map@k"]
].sort_values("metrics.ndcg@k", ascending=False)

print("\n=== Comparacao de Baselines ===")
print(comparison_df.to_string(index=False))

# ===== ARTEFATOS PARA DVC (Reprodutibilidade) =====
# Criar diretorio para dados processados
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

# Salvar datasets de treino/validacao/teste (para reprodutibilidade)
train_df.to_csv(processed_dir / "train.csv", index=False)
valid_df.to_csv(processed_dir / "valid.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

# Salvar recomendações dos baselines (artefatos de modelo)
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

import json
with open(models_dir / "dummy_recommendations.json", "w") as f:
    json.dump({str(u): v for u, v in dummy_recs.items()}, f)

with open(models_dir / "baseline_item_item_recommendations.json", "w") as f:
    json.dump({str(u): v for u, v in baseline_recs.items()}, f)

# Salvar métricas detalhadas por modelo
metrics_summary = {
    "dummy_metrics": dummy_metrics,
    "baseline_metrics": baseline_metrics,
    "comparison": comparison_df.to_dict(orient="records")
}
with open(models_dir / "baseline_metrics_detailed.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

# Salvar CSV para facilitar visualizacao
comparison_df.to_csv(models_dir / "baseline_experiments_comparison.csv", index=False)

print(f"\n✅ Artefatos salvos para versionamento DVC:")
print(f"   - {processed_dir / 'train.csv'}")
print(f"   - {processed_dir / 'valid.csv'}")
print(f"   - {processed_dir / 'test.csv'}")
print(f"   - {models_dir / 'dummy_recommendations.json'}")
print(f"   - {models_dir / 'baseline_item_item_recommendations.json'}")
print(f"   - {models_dir / 'baseline_metrics_detailed.json'}")
print(f"   - {models_dir / 'baseline_experiments_comparison.csv'}")

## Proximo passo

1. Ajustar hiperparametros (TOP_K, pesos de evento, janela temporal).
2. Repetir a avaliacao no conjunto de teste.
3. Levar o mesmo protocolo para o modelo neural em PyTorch e comparar com estes baselines.